# AQI Predictor — Lahore | Training Pipeline (multi-horizon)

Trains **direct multi-horizon** models: a separate best-model for AQI **+24h, +48h, +72h**,
so the dashboard can show a 3-day forecast (one point per day).

For each horizon it trains **Ridge**, **RandomForest**, **XGBoost**, and an **MLP** (PyTorch),
compares them against a **persistence baseline** on RMSE / MAE / R², and registers the best in
the **Model Registry** as `lahore_aqi_model_24h` / `_48h` / `_72h`.

> Expect accuracy to drop as the horizon grows — predicting 3 days out is genuinely harder
> than 1 day. That degradation is a real, reportable finding, not a bug.

## 0. Setup

In [1]:
!pip install scikit-learn xgboost torch joblib pandas numpy --quiet


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

import torch
import torch.nn as nn

# Predict AQI these many hours ahead — one model per horizon (one dashboard point per day).
HORIZONS = [24, 48, 72]

## 1. Connect and read from the Feature Store

Set your API key as an environment variable first (don't hardcode it):
PowerShell → `setx HOPSWORKS_KEY "your_key"`, then restart VS Code.

In [6]:
import hopsworks

project = hopsworks.login(
    project="project123456789",                   # your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_KEY"],
)

fs = project.get_feature_store()
fg = fs.get_feature_group("lahore_aqi_features", version=1)
df = fg.read()
print("rows read from feature store:", len(df))

# order matters for the chronological split and for shifting the target
df = df.sort_values("time").reset_index(drop=True)
df.head()

2026-09-06 02:20:27,654 INFO: Closing external client and cleaning up certificates.
2026-09-06 02:20:27,659 INFO: Connection closed.
2026-09-06 02:20:27,663 INFO: Initializing external client
2026-09-06 02:20:27,665 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-06 02:20:30,152 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/43139
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (78.63s) 
rows read from feature store: 17568


,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation,us_aqi,...,aqi_change_rate,aqi_lag_1h,pm25_lag_1h,aqi_lag_3h,pm25_lag_3h,aqi_lag_24h,pm25_lag_24h,aqi_roll_mean_24h,pm25_roll_mean_24h,aqi_roll_max_24h
0,2024-08-31 00:00:00+00:00,25.2,94,982.9,5.5,337,1,24.2,0.0,85,...,0.0,85.0,35.3,95.0,31.2,77.0,28.0,84.750000,27.604167,117.0
1,2024-08-31 01:00:00+00:00,25.1,95,982.7,4.0,350,1,24.3,0.0,86,...,1.0,85.0,34.6,84.0,33.9,77.0,26.5,85.083333,27.879167,117.0
2,2024-08-31 02:00:00+00:00,24.9,96,982.2,4.4,9,1,24.3,0.0,86,...,0.0,86.0,35.5,85.0,35.3,76.0,25.7,85.458333,28.254167,117.0
3,2024-08-31 03:00:00+00:00,24.8,97,981.9,4.7,360,0,24.3,0.0,87,...,1.0,86.0,35.6,85.0,34.6,76.0,25.4,85.875000,28.666667,117.0
4,2024-08-31 04:00:00+00:00,24.7,98,981.8,4.0,333,1,24.3,0.0,88,...,1.0,87.0,35.6,86.0,35.5,76.0,25.5,86.333333,29.091667,117.0


## 2. Feature columns (computed once)

The feature set is identical for every horizon, so select it once. Grabbing only numeric
columns automatically drops `time`, `city`, and `aqi_category`.

In [7]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != "target"]
print("features:", len(feature_cols))

features: 39


## 3. MLP definition (PyTorch)

Defined at top level (not inside the loop) so it can be re-instantiated later to load a saved
model. The target is standardized during training and un-standardized at prediction, which is
what keeps the MLP numerically stable.

In [8]:
class MLPRegressor(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train_s, y_train, epochs=150, seed=42):
    torch.manual_seed(seed)
    y_mean, y_std = float(y_train.mean()), float(y_train.std())

    Xtr = torch.tensor(X_train_s, dtype=torch.float32)
    ytr = torch.tensor(((y_train - y_mean) / y_std).values, dtype=torch.float32).view(-1, 1)

    model = MLPRegressor(X_train_s.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=32, shuffle=True
    )

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    return model, y_mean, y_std


def predict_mlp(model, X_s, y_mean, y_std):
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(X_s, dtype=torch.float32)).numpy().ravel()
    return p * y_std + y_mean          # inverse-transform back to real AQI units

## 4. Train-and-evaluate for one horizon

Everything horizon-specific lives here: the target shift, the chronological split (each
horizon drops a different number of trailing rows), the scaler, and all five evaluations.

In [9]:
def train_for_horizon(df, horizon, feature_cols):
    d = df.copy()
    d["target"] = d["us_aqi"].shift(-horizon)                 # AQI `horizon` hours ahead
    d = d[feature_cols + ["target"]].dropna().reset_index(drop=True)

    # chronological split — never random (rows are an ordered series with lag features)
    split = int(len(d) * 0.8)
    tr, te = d.iloc[:split], d.iloc[split:]
    X_train, y_train = tr[feature_cols], tr["target"]
    X_test,  y_test  = te[feature_cols], te["target"]

    # scale for the scale-sensitive models (Ridge, MLP); fit on TRAIN only
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    results = []
    def ev(name, y_true, y_pred):
        results.append({
            "horizon": horizon, "model": name,
            "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
            "mae":  mean_absolute_error(y_true, y_pred),
            "r2":   r2_score(y_true, y_pred),
        })

    # baseline: "AQI in `horizon`h = AQI now"
    ev("Persistence", y_test, X_test["us_aqi"])

    ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
    ev("Ridge", y_test, ridge.predict(X_test_s))

    rf = RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42
    ).fit(X_train, y_train)
    ev("RandomForest", y_test, rf.predict(X_test))

    xgb = XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42
    ).fit(X_train, y_train)
    ev("XGBoost", y_test, xgb.predict(X_test))

    mlp, y_mean, y_std = train_mlp(X_train_s, y_train)
    ev("MLP", y_test, predict_mlp(mlp, X_test_s, y_mean, y_std))

    res_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
    trained = {"Ridge": ridge, "RandomForest": rf, "XGBoost": xgb, "MLP": mlp}
    mlp_meta = {"y_mean": y_mean, "y_std": y_std, "input_dim": X_train_s.shape[1]}
    return res_df, trained, scaler, mlp_meta

## 5. Loop over horizons — train, compare, register

For each horizon: run the four evaluations, pick the best real model (excluding the
persistence baseline), save it plus the scaler and feature order, and register it.

In [10]:
mr = project.get_model_registry()
all_results = []

for h in HORIZONS:
    res_df, trained, scaler, mlp_meta = train_for_horizon(df, h, feature_cols)
    all_results.append(res_df)
    print(f"\n=== horizon +{h}h ===")
    print(res_df.to_string(index=False))

    # best actual model — persistence is a baseline, not something to register
    best = res_df[res_df.model != "Persistence"].iloc[0]
    best_name = best["model"]
    best_model = trained[best_name]

    model_dir = f"aqi_model_{h}h"
    os.makedirs(model_dir, exist_ok=True)

    if best_name == "MLP":
        torch.save(best_model.state_dict(), os.path.join(model_dir, "model.pt"))
        joblib.dump(mlp_meta, os.path.join(model_dir, "mlp_meta.pkl"))   # needed to rebuild + unscale
    else:
        joblib.dump(best_model, os.path.join(model_dir, "model.pkl"))

    joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))
    joblib.dump(feature_cols, os.path.join(model_dir, "feature_cols.pkl"))

    m = mr.python.create_model(
        name=f"lahore_aqi_model_{h}h",
        metrics={"rmse": float(best["rmse"]), "mae": float(best["mae"]), "r2": float(best["r2"])},
        description=f"Predicts Lahore US AQI +{h}h ahead — best: {best_name}",
    )
    m.save(model_dir)
    print(f"registered lahore_aqi_model_{h}h  ({best_name})")


=== horizon +24h ===
 horizon        model      rmse       mae       r2
      24      XGBoost 25.364616 18.378310 0.589398
      24 RandomForest 25.738276 18.633481 0.577212
      24        Ridge 26.463401 19.426130 0.553054
      24          MLP 29.064371 20.675920 0.460879
      24  Persistence 33.141277 20.271302 0.299025


  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'aqi_model_24h' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading c:\Users\theam\Internship - 10Pearls\aqi_model_24h/feature_cols.pkl: 0.000%|          | 0/582 elapse…

Uploading c:\Users\theam\Internship - 10Pearls\aqi_model_24h/model.pkl: 0.000%|          | 0/805246 elapsed<00…

...

Moving model files from 'aqi_model_72h' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading c:\Users\theam\Internship - 10Pearls\aqi_model_72h/feature_cols.pkl: 0.000%|          | 0/582 elapse…

Uploading c:\Users\theam\Internship - 10Pearls\aqi_model_72h/model.pkl: 0.000%|          | 0/864 elapsed<00:00…

Uploading c:\Users\theam\Internship - 10Pearls\aqi_model_72h/scaler.pkl: 0.000%|          | 0/2559 elapsed<00:…

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/43139/models/lahore_aqi_model_72h/2
registered lahore_aqi_model_72h  (Ridge)


## 6. Combined comparison across horizons

In [11]:
comparison = pd.concat(all_results, ignore_index=True)
comparison

,horizon,model,rmse,mae,r2
0,24,XGBoost,25.364616,18.378310,0.589398
1,24,RandomForest,25.738276,18.633481,0.577212
2,24,Ridge,26.463401,19.426130,0.553054
3,24,MLP,29.064371,20.675920,0.460879
4,24,Persistence,33.141277,20.271302,0.299025
5,48,Ridge,33.814976,25.383168,0.270292
6,48,Persistence,39.008392,25.103881,0.028938
7,48,MLP,42.573800,31.183850,-0.156686
8,48,XGBoost,42.967139,31.612997,-0.178158
9,48,RandomForest,43.822100,31.330222,-0.225511
